In [1]:
import os
from pypdf import PdfWriter

# Create dummy TXT
with open("test.txt", "w") as f:
    f.write("This is a standard text document.\nIt contains plain text.")

# Create dummy MD
with open("test.md", "w") as f:
    f.write("# Markdown File\n\nThis is a dummy markdown file.\n\n## Section 1\nContent here.")

# Create dummy PDF using pypdf
writer = PdfWriter()
writer.add_blank_page(width=200, height=200)
with open("test.pdf", "wb") as f:
    writer.write(f)

print("Dummy files generated: test.txt, test.md, test.pdf")


Dummy files generated: test.txt, test.md, test.pdf


# Day 27: Refactoring the LangChain Pipeline for Multiple Document Types

Welcome to Day 27! Today we elevate our RAG (Retrieval-Augmented Generation) pipeline from a simple text processor to a robust, production-ready document ingestion engine capable of handling varied real-world data sources (PDFs, Markdown, and TXT files).

## The "Why" and "How"

**The Problem:**
Most production systems do not ingest nicely formatted text strings. Enterprise knowledge bases contain unstructured PDFs, documentation in Markdown, and raw log files in TXT. Hardcoding a single loader (like `TextLoader`) breaks the moment a user uploads a different file type.

**The Solution:**
We need to implement a **Factory Pattern** or a **routing mechanism** within our pipeline. 
1. **Dynamic Selection:** We inspect the file extension and dynamically instantiate the correct LangChain DocumentLoader.
2. **Unified Output:** Regardless of the underlying loader (`PyPDFLoader`, `UnstructuredMarkdownLoader`, `TextLoader`), the output must always be a standardized list of LangChain `Document` objects containing `page_content` and `metadata`. This ensures the downstream pipeline (chunking, embedding, vector DB ingestion) remains completely agnostic to the source format.

## Core Theory (Just-in-Time)

In a production Retrieval-Augmented Generation (RAG) system, user data rarely comes in a single uniform format. Organizations rely on text files (.txt), Markdown documentation (.md), and portable document formats (.pdf), among others. Refactoring our pipeline to ingest multiple formats dynamically is essential for building a robust Knowledge Base.

### The "Why" and "How"
- **Why?** Hardcoding loaders for specific file types limits system flexibility. A generic ingestion pipeline allows you to drop any file into a directory and have the system automatically select the appropriate parser.
- **How?** We use a factory or mapping pattern to dynamically route a file to its correct loader class (e.g., `TextLoader` for .txt, `PyPDFLoader` for .pdf) based on its file extension.

### AI Security & Production Readiness
1. **PII and Data Sensitivity:** When parsing varied document types (especially PDFs from external sources), you risk ingesting Personally Identifiable Information (PII) or sensitive credentials. While parsing, always consider adding a redaction or filtering layer before documents are embedded.
2. **Fallback Mechanisms:** File parsing is notoriously brittle. A PDF might be corrupted, or a text file might have unexpected encodings. Your ingestion logic must implement graceful error handling (`try/except` blocks) to skip corrupted files without crashing the entire batch process.
3. **Malicious Payloads:** Documents can contain embedded scripts or malformed content designed to trigger denial-of-service (e.g., infinite loops during parsing). Always validate file sizes and use timeouts when processing external files.

## Code Implementation: Multi-Document Pipeline

Below is a tiered progression of refactoring the document ingestion pipeline. We move from a hardcoded basic approach to a robust, object-oriented, production-ready system.

### Basic: Isolate the Core Concept
This example demonstrates the minimal logic needed to route files to different loaders based on an `if/else` block. It lacks robust error handling and OOP structure.

In [2]:
from typing import List
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
import os

def load_document_basic(file_path: str) -> List[Document]:
    _, ext = os.path.splitext(file_path)
    ext = ext.lower()
    
    if ext == ".txt" or ext == ".md":
        loader = TextLoader(file_path)
    elif ext == ".pdf":
        loader = PyPDFLoader(file_path)
    else:
        print(f"Unsupported extension: {ext}")
        return []
        
    return loader.load()

# Test Basic
if os.path.exists('test.txt'):
    print(f"Basic loaded test.txt: {len(load_document_basic('test.txt'))} docs")


/tmp/ipykernel_25129/2089829787.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


Basic loaded test.txt: 1 docs


### Medium: Clean OOP and State Management

Here, we encapsulate the logic within a class, utilizing a dictionary to map extensions to loader classes. This makes the code extensible without modifying the core logic `if/else` chain.

In [3]:
from typing import List, Dict, Type
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.document_loaders.base import BaseLoader
import os

class DocumentLoaderMedium:
    def __init__(self) -> None:
        # Mapping extensions to classes (Factory pattern)
        self.loaders: Dict[str, Type[BaseLoader]] = {
            ".txt": TextLoader,
            ".md": TextLoader,
            ".pdf": PyPDFLoader
        }
        
    def load(self, file_path: str) -> List[Document]:
        _, ext = os.path.splitext(file_path)
        loader_cls = self.loaders.get(ext.lower())
        
        if not loader_cls:
            print(f"Warning: No loader found for {ext}")
            return []
            
        loader = loader_cls(file_path)
        return loader.load()

# Test Medium
ingestor_med = DocumentLoaderMedium()
if os.path.exists('test.pdf'):
    print(f"Medium loaded test.pdf: {len(ingestor_med.load('test.pdf'))} docs")


Medium loaded test.pdf: 1 docs


### Advanced: Production-Grade Implementation

This version is interview-ready and production-safe. It features strict type hinting, comprehensive docstrings, robust error handling with fallback mechanisms, and explicit AI security considerations (e.g., catching file errors and preventing complete batch failures).

In [4]:
import os
import logging
from typing import List, Dict, Type, Optional
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.document_loaders.base import BaseLoader

# Configure basic logging for production visibility
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DocumentIngestor:
    """
    A robust pipeline component for ingesting multiple document types into LangChain Documents.
    Demonstrates clean OOP, dynamic loading, and fallback error handling for production environments.
    """
    
    def __init__(self) -> None:
        """Initializes the ingestor with a mapping of extensions to LangChain loaders."""
        self.loader_mapping: Dict[str, Type[BaseLoader]] = {
            ".txt": TextLoader,
            ".md": TextLoader,
            ".pdf": PyPDFLoader
        }
        
    def load_document(self, file_path: str) -> List[Document]:
        """
        Dynamically selects the appropriate loader based on the file extension and safely loads the document.
        
        Args:
            file_path (str): The absolute or relative path to the file.
            
        Returns:
            List[Document]: A list of LangChain Document objects. Returns an empty list on failure (Fallback).
            
        Raises:
            ValueError: If the file does not exist, to prevent downstream processing of ghost files.
        """
        if not os.path.exists(file_path):
            logger.error(f"Security/Validation Error: File not found at path: {file_path}")
            raise ValueError(f"File not found: {file_path}")
            
        _, ext = os.path.splitext(file_path)
        ext = ext.lower()
        
        loader_cls: Optional[Type[BaseLoader]] = self.loader_mapping.get(ext)
        if not loader_cls:
            logger.warning(f"Unsupported file extension: {ext}. Skipping file: {file_path}")
            # Fallback mechanism: Return empty list instead of crashing the batch process
            return []
            
        logger.info(f"Ingesting {file_path} using {loader_cls.__name__}...")
        try:
            loader = loader_cls(file_path)
            # In a real production scenario with massive files, consider loader.lazy_load()
            documents: List[Document] = loader.load()
            
            # Simulated Security Step: Add metadata tagging or PII redaction checks here
            for doc in documents:
                doc.metadata["source_type"] = ext
                
            return documents
        except Exception as e:
            # Fallback mechanism: Catch parsing errors (e.g., corrupted PDF) gracefully
            logger.error(f"Failed to parse {file_path}. Reason: {str(e)}")
            return []

# Test Advanced
if __name__ == "__main__":
    ingestor = DocumentIngestor()
    test_files = ["test.txt", "test.md", "test.pdf", "missing_file.docx"]
    
    all_docs: List[Document] = []
    for file in test_files:
        try:
            if os.path.exists(file) or file == "missing_file.docx":
                docs = ingestor.load_document(file)
                all_docs.extend(docs)
        except ValueError as e:
            logger.error(f"Batch processing error handled: {e}")
            
    print(f"\nTotal documents successfully loaded: {len(all_docs)}")


2026-08-22 08:07:00,708 - INFO - Ingesting test.txt using TextLoader...


2026-08-22 08:07:00,710 - INFO - Ingesting test.md using TextLoader...


2026-08-22 08:07:00,711 - INFO - Ingesting test.pdf using PyPDFLoader...


2026-08-22 08:07:00,716 - ERROR - Security/Validation Error: File not found at path: missing_file.docx


2026-08-22 08:07:00,716 - ERROR - Batch processing error handled: File not found: missing_file.docx



Total documents successfully loaded: 3


## Common Pitfalls in Production

1. **Ignoring Metadata Normalization:** Different loaders extract different metadata. For example, `PyPDFLoader` includes the `page` number, while `TextLoader` does not. Downstream components (like Qdrant payload filters) will crash if they expect a `page` key that doesn't exist. Always normalize metadata after loading.
2. **Memory Overload with Large Files:** Calling `.load()` on a 10,000-page PDF loads the entire document into memory. For large files, use `.lazy_load()` which returns an iterator, preventing OOM (Out Of Memory) exceptions.
3. **Missing System Dependencies:** Loaders like `PyPDFLoader` (requires `pypdf`) rely on underlying system libraries. Forgetting to include these in your `requirements.txt` or Dockerfile is a classic deployment failure.
4. **Failing Open on Errors:** If a file parser crashes, a poorly designed system might abort the entire batch ingestion. Implementing fallback mechanisms (returning empty lists or logging and skipping) ensures partial success is retained.

## Practical Lab / Homework

**Your Task:**
1. Create a function `process_directory(directory_path: str, ingestor: DocumentIngestor) -> List[Document]`.
2. The function should iterate through all files in the given directory.
3. It should attempt to load each file using the `ingestor` instance.
4. It must catch `ValueError` for missing files gracefully and utilize the fallback mechanisms built into the advanced ingestor to skip unsupported files.
5. Return the combined list of all successfully loaded `Document` objects.

*Challenge: Record a brief async video walkthrough (Loom/etc.) of your design decisions, specifically explaining how you handled the fallback mechanisms.*

In [5]:
# Lab Implementation
def process_directory(directory_path: str, ingestor: DocumentIngestor) -> List[Document]:
    """
    Scans a directory and safely ingests all supported documents.
    """
    combined_docs: List[Document] = []
    
    print(f"Scanning directory: {directory_path}")
    for filename in os.listdir(directory_path):
        filepath = os.path.join(directory_path, filename)
        
        # Skip directories
        if os.path.isdir(filepath):
            continue
            
        try:
            docs = ingestor.load_document(filepath)
            if docs:
                combined_docs.extend(docs)
        except ValueError as e:
            print(f"Directory scanner caught error: {e}")
            
    return combined_docs

# Run the Lab
final_docs = process_directory(".", DocumentIngestor())
print(f"\nLab Complete! Total documents ingested from directory: {len(final_docs)}")


2026-08-22 08:07:00,729 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_31.ipynb


2026-08-22 08:07:00,730 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_48.ipynb


2026-08-22 08:07:00,731 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_47.ipynb


2026-08-22 08:07:00,732 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_40.ipynb


2026-08-22 08:07:00,733 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_21.ipynb


...

2026-08-22 08:07:00,776 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_30.ipynb


2026-08-22 08:07:00,777 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_42.ipynb


2026-08-22 08:07:00,778 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_38.ipynb


2026-08-22 08:07:00,778 - WARNING - Unsupported file extension: .ipynb. Skipping file: ./day_11.ipynb


Scanning directory: .

Lab Complete! Total documents ingested from directory: 3


## Reference Links

- [LangChain Document Loaders Documentation](https://python.langchain.com/docs/modules/data_connection/document_loaders/)
- [LangChain PyPDFLoader Specifics](https://python.langchain.com/docs/modules/data_connection/document_loaders/pdf/)
- [Python Design Patterns: Factory Method](https://refactoring.guru/design-patterns/factory-method/python/example)